# 01 - HMI FITS exploration

Load HMI frames with `astropy`, inspect headers, disk geometry and value distributions.

Download data first with `scripts/download_hmi.py`; without data the notebook falls back to a synthetic sequence.

In [ ]:
%matplotlib inline

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from hmi_infill import prep, synthetic, viz

FITS_DIR = Path("../data/raw/hmi.M_45s")   # change to the series you downloaded
CROP = None                                 # e.g. 1024 for faster experiments
paths = sorted(FITS_DIR.glob("*.fits"))
if paths:
    loaded = [prep.load_hmi_int(p, crop=CROP) for p in paths[:6]]
    frames, valids, info, header = [l[0] for l in loaded], [l[1] for l in loaded], loaded[0][2], loaded[0][3]
    print(f"{len(frames)} FITS frames from {FITS_DIR}, shape {frames[0].shape}")
else:
    print("No FITS found -> synthetic magnetogram sequence")
    conv = [prep.to_int(f, content="MAGNETOGRAM") for f in synthetic.sequence(6, 512, "magnetogram", seed=0)]
    frames, valids, info, header = [c[0] for c in conv], [c[1] for c in conv], conv[0][2], None
print(info)

## Header

In [ ]:
if header is not None:
    for k in ("T_OBS", "CONTENT", "BUNIT", "CDELT1", "CRPIX1", "CRPIX2", "CRVAL1", "CRVAL2", "CROTA2", "RSUN_OBS", "R_SUN", "DSUN_OBS", "QUALITY", "MISSVALS"):
        print(f"{k:10s}", header.get(k))

## Image, disk mask and centre-to-limb geometry

In [ ]:
phys = prep.from_int(frames[0], info, valids[0])
fig, axs = plt.subplots(1, 3, figsize=(17, 5))
viz.show(phys, axs[0], "frame 0 (physical units)", norm=viz.norm_for(phys, percentile=99.5, symmetric=info.content.upper() != "CONTINUUM INTENSITY"))
viz.show(valids[0].astype(float), axs[1], "valid (finite) pixels", colorbar=False)
if header is not None:
    viz.show(prep.mu_map(header, phys.shape), axs[2], "mu = cos(theta)")
else:
    axs[2].axis("off")
plt.tight_layout()

## Value distribution and frame-to-frame differences

The difference between consecutive frames shows how much temporal redundancy the infill can exploit.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 4.5))
v = phys[np.isfinite(phys)]
axs[0].hist(v, bins=400, log=True); axs[0].set_title("histogram (on-disk)"); axs[0].set_xlabel(info.content or "value")
d = (frames[1].astype(float) - frames[0]) * info.scale
d[~(valids[0] & valids[1])] = np.nan
viz.show(d, axs[1], "frame 1 - frame 0", "RdBu_r", viz.norm_for(d, symmetric=True))
plt.tight_layout()
for i in range(1, len(frames)):
    m = valids[i] & valids[i - 1]
    diff = (frames[i].astype(float) - frames[i - 1])[m] * info.scale
    print(f"frame {i}-{i-1}: rms diff {diff.std():.2f}, rms signal {(frames[i][m] * info.scale).std():.2f}")